In [1]:
!git clone https://github.com/villerbond/avito-text-orientation.git
%cd avito-text-orientation

Cloning into 'avito-text-orientation'...
remote: Enumerating objects: 78, done.
remote: Total 78 (delta 0), reused 0 (delta 0), pack-reused 78 (from 1)
Receiving objects: 100% (78/78), 42.42 MiB | 19.29 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/avito-text-orientation


In [24]:
import sys
import os
from pathlib import Path
import requests
import torch
import zipfile
from torch.utils.data import DataLoader, Dataset
from PIL import Image

from src.utils import set_seed, get_device
from src.data import get_val_transform
from src.models import build_model

In [3]:
PROJECT_DIR = Path.cwd()
while not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = PROJECT_DIR.parent

In [9]:
set_seed(42)
device = get_device()

In [7]:
TEST_DIR = PROJECT_DIR / "data" / "test"
TEST_DIR.mkdir(parents=True, exist_ok=True)

In [11]:
YANDEX_URL = "https://disk.360.yandex.ru/d/ha0Q70T7ie-0_w"
ARCHIVE_PATH = PROJECT_DIR / "data" / "test_data.zip"

def download_yandex_file(public_url, save_path):
    api = "https://cloud-api.yandex.net/v1/disk/public/resources/download"

    download_url = requests.get(
        api,
        params={"public_key": public_url}
    ).json()["href"]

    with requests.get(download_url, stream=True) as r:
        r.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)

    print(f"Saved to {save_path}")


if not ARCHIVE_PATH.exists():
    download_yandex_file(YANDEX_URL, ARCHIVE_PATH)
else:
    print("Archive already exists.")

Saved to /content/avito-text-orientation/data/test_data.zip


In [15]:
if not any(TEST_DIR.iterdir()):
    with zipfile.ZipFile(ARCHIVE_PATH, "r") as z:
        z.extractall(TEST_DIR)
    print("Archive extracted.")
else:
    print("Data already extracted.")

Archive extracted.


In [16]:
TEST_IMAGES_DIR = TEST_DIR / "test" / "images"

In [17]:
TEST_IMAGES_DIR

PosixPath('/content/avito-text-orientation/data/test/test/images')

In [22]:
class InferenceDataset(Dataset):
    def __init__(self, image_paths: list[Path], transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        path = self.image_paths[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        image_id = path.stem
        return image, image_id


def build_test_loader(test_images_dir: Path, batch_size: int = 128, num_workers: int = 2):
    image_paths = sorted(test_images_dir.glob("*.png")) + sorted(test_images_dir.glob("*.jpg"))
    dataset = InferenceDataset(image_paths, transform=get_val_transform())
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    return loader

In [21]:
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm


@torch.no_grad()
def predict(model, loader, device) -> pd.DataFrame:
    """Прогоняет модель по loader, возвращает DataFrame [image_id, p_180]."""
    model.eval()
    image_ids = []
    probabilities = []

    for images, ids in tqdm(loader, desc="Inference"):
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy().flatten()

        image_ids.extend(ids)
        probabilities.extend(probs)

    return pd.DataFrame({"image_id": image_ids, "p_180": probabilities})


def validate_submission(submission_df: pd.DataFrame, sample_submission_path: Path) -> None:
    """Сверяет структуру submission.csv с sample_submission.csv."""
    sample_df = pd.read_csv(sample_submission_path)

    assert list(submission_df.columns) == ["image_id", "p_180"], "Неверные колонки"
    assert len(submission_df) == len(sample_df), (
        f"Число строк не совпадает: {len(submission_df)} vs {len(sample_df)}"
    )
    assert submission_df["p_180"].between(0, 1).all(), "p_180 вне диапазона [0, 1]"
    assert submission_df["p_180"].isna().sum() == 0, "Есть пропущенные значения"

    missing_ids = set(sample_df["image_id"]) - set(submission_df["image_id"])
    assert not missing_ids, f"Отсутствуют image_id: {list(missing_ids)[:5]}..."

    print("Проверка пройдена: формат submission.csv корректен.")

In [23]:
BEST_MODEL_NAME = "resnet18"
CHECKPOINT_PATH = PROJECT_DIR / "checkpoints" / f"{BEST_MODEL_NAME}_best.pth"

In [25]:
model = build_model(BEST_MODEL_NAME).to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

print(f"Модель {BEST_MODEL_NAME} загружена из {CHECKPOINT_PATH}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 300MB/s]


Модель resnet18 загружена из /content/avito-text-orientation/checkpoints/resnet18_best.pth


In [26]:
test_loader = build_test_loader(TEST_IMAGES_DIR, batch_size=32)

In [ ]:
submission_df = predict(model, test_loader, device)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Inference:   0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
print(submission_df.shape)
submission_df.head()